Help : https://cengel.github.io/R-spatial/spatialops.html

In [ ]:
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png r::r-raster conda-forge::r-fnn")
system("conda install -y conda-forge::r-cluster conda-forge::r-remotes conda-forge::r-devtools")
system("conda install -y conda-forge::r-factominer conda-forge::r-caret conda-forge::r-factoextra conda-forge::r-rlang")
system("conda install -y conda-forge::r-geojsonio")


In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse) # because who can live without the tidyverse?
library(terra)     
library(dplyr)
        
library(jsonlite) 
library(utils)
library(ggplot2)

library(ncdf4) #     ncdf4: open, write and create NetCDF files (also provides metadata information)
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

library(data.table)
# library(FNN)

library(cluster)
# library(bioregion)

# library(rlang)
# library(factoextra)

library(sf)
library(geojsonio)

In [ ]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"Data",sep = "/")
outputDir <- paste(workdir,"outputs/collection",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.filename <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
    } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename, type = "gz"){
    filename.length <- nchar(filename)
    print(filename, filename.length)

    start <- 1

    if(type == "gz"){
        # Case Gunzip
        end <- filename.length-3
    }
    else{
        # Case Unzip
        end <- filename.length-4
    }

    unzip.filename <- substr(filename,start, end)
    print(unzip.filename)

    
    # Case gunzip
    if(!file.exists(unzip.filename)){
        if(type == "gz"){
            R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
            }
        else{
            utils::unzip(filename, overwrite=FALSE )
            }
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}


In [ ]:
gdb_path_zip <- "Geomorphology.gdb.zip"

download.filename(url = "https://d28rz98at9flks.cloudfront.net/102441/Geomorphology.gdb.zip",
                 filename = gdb_path_zip) # https://data.gov.au/data/dataset/geomorphic-features-of-the-antarctic-margin-and-southern-ocean-20126/resource/cedf76d9-4390-4080-b9d0-84c57bd1c7b8

gdb_path <- unzip.file(gdb_path_zip, type = "zip")

# gdb_path <- move.file(gdb_path, paste(dataDir,"Geomorphology.gdb",sep = "/"))


In [ ]:
#Correct unzipped path
gdb_path <- unlist(strsplit(gdb_path, "/"))[[length( unlist(strsplit(gdb_path, "/")) )]]
print(gdb_path)

In [ ]:
# https://gis.stackexchange.com/questions/426282/load-gdb-directory-into-r-using-simple-features-package
model <- sf::st_read(dsn = gdb_path)

# Select Seamount, Seamount Ridges and Canyon
seamounts <- model$Feature[model$Feature %in% c("Seamount", "Seamount Ridges")]
canyons <- model$Feature[model$Feature %in% c("Canyon")]
others <- model$Feature[model$Feature %in% c("Coastal/Shelf Terrane", "Cross Shelf Valley",
                                            "Ridge", "Shelf Deep", "Trough Mouth Fan", 
                                            "Contourite Feature", "Plateau", "")]


In [ ]:
print(model)


In [ ]:
unique(model$Feature)
table(model$Feature)
head(model$Shape_Area,5)
head(model$Shape_Length,5)


In [ ]:
model$Shape

In [ ]:
## HELP
# https://r.geocompx.org/solutions/read-write.html
# https://github.com/highered-esricanada/r-arcgis-tutorials/blob/master/3-R-ArcGIS-Scripting.pdf
# = https://esricanada-ce.github.io/r-arcgis-tutorials/3-R-ArcGIS-Scripting.pdf


In [ ]:
# View
plot(model)


In [ ]:
# List layers inside the geodatabase
gbd.layers <- st_layers(gdb_path)
print(gbd.layers)

In [ ]:
# Analysis
summary(model)

In [ ]:
# Check for the projection
st_crs(model)

In [ ]:
plot(st_geometry(model), border="#aaaaaa", main="Census tracts around city center,\nclipped by 2km buffer ")

st_crs(model)$proj4string

plot(st_geometry(model), pch = 20, col="#ccc", 
     axes=F, main = "before transform - WGS84")
axis(side = 1, las = 3) # adjust text on axes
axis(side = 2, las = 1)


In [ ]:
ext(model)
ncell(model)

In [ ]:
#st_point(c(1750160, 467499.9)) %>% # point coordinates
#  st_sfc(crs = st_crs(model))  # create feature collection, setting CRS to philly_sf's CRS


model_ctr <- st_point(c(1750160, 467499.9)) %>%
    st_sfc(crs = st_crs(model))
st_crs(model_ctr)$proj4string
model_buf <-  st_buffer(model_ctr, 10000)
model_sel <- st_filter(model, model_buf)


In [ ]:
plot(st_geometry(model), border="#aaaaaa", main="Census tracts around city center,\nclipped by 2km buffer ")
plot(st_geometry(model_sel), add=T, col="red")
plot(st_geometry(model_buf), add=T, lwd = 2)


In [ ]:
model_intersection <- st_intersection(model_buf, model)
model_intersection

plot(st_geometry(model), border="#aaaaaa", main="Census tracts around city center,\nclipped by 2km buffer ")
plot(model_intersection, add=T, lwd = 2, border = "red")


In [ ]:
# If it's not WGS84 (EPSG:4326)

forms_to_coords <- function(model_sf){
    # data <- st_transform(model, 4326)
    # data <- st_make_valid(data)
    data <- st_make_valid(model_sf)
    
    # ⚠️ If geometry is NOT points
    #🔸 For polygons or lines
    centroids <- st_centroid(data)
    coords <- st_coordinates(centroids)
    return(coords)
}


In [ ]:
# Export to 
# write.csv(st_drop_geometry(data), paste("outputs/collection/st_drop_geometry.csv"), row.names = FALSE)

coords <- forms_to_coords(model)
head(coords, 5)

In [ ]:
range(coords)

In [ ]:
# If it's not WGS84 (EPSG:4326)

forms_to_coords <- function(model_sf, std.WGS84 = TRUE){
    coords <- st_transform(model_sf, 4326) %>%
        st_make_valid() %>%
        st_centroid() %>%
        st_coordinates()
    colnames(coords) <- c("lon","lat")
    return(coords)
}


In [ ]:
# ==============================================================================
# 1. COMPUTE DISTANCE TO CANYON/ SEAMOUNT
# ==============================================================================

# All points
plot(coords)

# Split canyon vs other features
# Canyon = Cross Shelf Valley
canyons <- model[model$Feature == "Canyon", ]
canyon_coords <- forms_to_coords(canyons)
plot(canyon_coords,add=T, pch = 3, col="black")


# Split seamounts vs other features
seamount_ridges <- model[model$Feature == "Seamount Ridges", ]
seamount_ridges_coords <- forms_to_coords(seamount_ridges)
# plot(seamount_ridges_coords,add=T, col="blue")

# Split seamount ridges vs other features
seamounts <- model[model$Feature == "Seamount", ]
seamounts_coords <- forms_to_coords(seamounts)
# plot(seamounts_coords,add=T, col="green")

# Split seamounts vs other features
seamount_and_Sridges <- model[model$Feature %in% c("Seamount Ridges", "Seamount"), ]
seamount_and_Sridges_coords <- forms_to_coords(seamount_and_Sridges)
points(seamount_and_Sridges_coords, pch = 16, col ="red")



In [ ]:

# Load your data (gdb layer)
v <- vect("Geomorphology.gdb", layer=gbd.layers$name)

# Reproject to lat/lon
v_ll <- project(v, "EPSG:4326")

# Extract coordinates
# Cooordinates ONLY
coords <- crds(v_ll)
# Other features INCLUDED
coords.and.feat <- as.data.frame(v, geom=TRUE)
head(coords) ; head(coords.and.feat)

In [ ]:
head(canyon_coords) ; plot(canyon_coords)

In [ ]:
# Compute distances
# Other features (or all, depending on your goal)
dist_matrix <- st_distance(coords, canyon_coords)
min_dist <- apply(dist_matrix, 1, min)

In [ ]:
# Extract minimum distance
other$dist_to_canyon <- as.numeric(min_dist)

In [ ]:
head(min_dist)

In [ ]:
forms_to_points <- function(model_sf){
    model_sf %>%
        st_transform(4326) %>%
        st_make_valid() %>%
        st_centroid()
}

In [ ]:
points_all <- forms_to_points(model) %>%
    st_transform(3031)

canyon_pts <- forms_to_points(canyons) %>%
    st_transform(3031)
seamount_pts <- forms_to_points(seamount_and_Sridges) %>%
    st_transform(3031)


In [ ]:
nearest_idx <- st_nearest_feature(points_all, canyon_pts)

min_dist <- st_distance(
  points_all,
  canyon_pts[nearest_idx, ],
  by_element = TRUE
)

points_all$dist_to_canyon <- as.numeric(min_dist)

nearest_idx <- st_nearest_feature(points_all, seamount_pts)

min_dist <- st_distance(
  points_all,
  seamount_pts[nearest_idx, ],
  by_element = TRUE
)

points_all$dist_to_seamount <- as.numeric(min_dist)

In [ ]:
# points_all$dist_to_canyon
# points_all$dist_to_seamount
# points_all$Feature

plot(points_all)

In [ ]:
# 2.2 DOWNLOAD AND LOAD BATHYMETRY ----
options(timeout = 600)  # 10 minutes
bathy_file <- "global_topo_1min_topo_19_1.nc"
if(file.exists(bathy_file)){
    cat(bathy_file, "is (are) already in your repertory.")
} else {
    download.file("https://topex.ucsd.edu/pub/global_topo_1min/topo_19.1.nc", bathy_file, mode = "wb")
    print('File Downloaded')
}

# MOVE TO DATA DIRECTORY 
ifelse(!dir.exists(file.path(dataDir)),
        dir.create(file.path(dataDir)),
        "Directory Exists")

file.rename(from=bathy_file,
            to=paste(dataDir, bathy_file, sep = "/"))
bathy_file <- paste(dataDir, bathy_file, sep = "/")

In [ ]:
# example: depth_raster

depth_raster <- rast(bathy_file)
depth_raster <- depth_raster[["z"]]   # adapt name
crs(depth_raster)
